# Day 2 · 视觉编码器：ViT → SigLIP

**配套讲义**: [`days/day-02.md`](../days/day-02.md) ｜ **需要 GPU（云机器）**

亲手写出 ViT 的 patch embedding + attention block，跑通 `(B,3,448,448)` → `(B,N,D)`，并说清 N 是怎么算出来的、SigLIP 比 CLIP 改了什么。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w1.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu    : {p.name}  {p.total_memory / 1024**3:.0f} GB")
    print("bf16   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️  没有 GPU —— 这一天的训练/推理跑不了。先看 docs/13-hardware-and-cost.md 租机器")

## 1. 手算 N，再用代码验证

**先算再跑** —— 直接跑代码你学不到东西。

In [ ]:
import math
for size, patch, merge in [(448, 14, 1), (448, 14, 2), (1024, 14, 2), (768, 14, 2)]:
    grid = math.ceil(size / patch)
    # merge=2 时，2×2 的相邻 patch 会被合并成一个 token → 数量除 4
    n = (grid // merge) ** 2
    print(f"{size}×{size}  patch={patch} merge={merge}  grid={grid}  N={n}  "
          f"（占 2048 序列的 {n/2048:.0%}）")
print("\n→ 注意最后那列：1024² 的图会吃掉序列的一大半，这就是「视觉 token 经济」")

## 2. 跑手写 ViT，逐层看 shape

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.minivlm.vision"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout or r.stderr)

## 3. 动手改坏它，看会发生什么

学习最快的方式是**故意写错**，然后观察症状。把下面几处各改一次（改完记得改回来）：

In [ ]:
# 这段是"实验设计"，不是要你运行它
experiments = [
    ("把 scale 从 head_dim**-0.5 改成 d_model**-0.5",
     "能跑，不报错 —— 但注意力分布会被压平，效果变差。这就是它难查的原因"),
    ("把 Pre-LN 改成 Post-LN",
     "浅层看不出问题，堆到 27 层 loss 会震荡或直接 NaN"),
    ("去掉位置编码",
     "shape 完全一样，模型却失去空间概念 —— 打乱 patch 顺序结果不变"),
    ("把 patch 从 14 改成 16",
     "448/16=28 → N=784。视觉 token 少了，但和预训练权重不再匹配"),
]
for change, symptom in experiments:
    print(f"改：{change}")
    print(f"  → {symptom}\n")
print("挑一个真的动手改一遍。看着代码在 shape 完全正确的情况下变差，是会记住的")

## 4. 打卡

In [ ]:
print("""今日打卡
─────────────────────────────────────────
[学到] N 的算法是 ______；Pre-LN 和 Post-LN 的差别是 ______
[产出] src/minivlm/vision.py 通过自检
[卡住] ______
─────────────────────────────────────────""")

## 验收清单

- [ ] 能**手算**任意尺寸图片的 N（例如 1024×768、patch=14、merge=2 → 答案见讲义）
- [ ] `shape_report()` 打印的每一层形状都能解释清楚
- [ ] 能说出 SigLIP 与 CLIP 的两点差异（loss 形式 / 是否用 softmax 归一化）
- [ ] 能回答「为什么 Qwen2.5-VL 用 SigLIP 而不是 CLIP」

**卡住了？** 回看 [`days/day-02.md`](../days/day-02.md) 第五节「容易踩的坑」。

> **明天**：`days/day-03.md` —— 连接器：为什么 LLaVA 用最笨的 MLP 反而赢了